In [137]:
import pandas as pd

silver_customers = pd.read_parquet(r"..\data\silver\olist_customers_dataset.parquet")
silver_orders = pd.read_parquet(r"..\data\staging\olist_orders_dataset.parquet")
silver_order_items = pd.read_parquet(
    r"..\data\silver\olist_order_items_dataset.parquet"
)
silver_products =  pd.read_parquet(r"..\data\silver\olist_products_dataset.parquet")
silver_payments = pd.read_parquet(r"..\data\silver\olist_order_payments_dataset.parquet")
silver_reviews = pd.read_parquet(r"..\data\silver\olist_order_reviews_dataset.parquet")

In [138]:
# Articles au niveau commande
items_agg = silver_order_items.groupby("order_id", as_index=False).agg(
    nb_items=("order_item_id", "count"),
    products_value=("price", "sum"),
    freight_value=("freight_value", "sum"),
    nb_sellers = ('seller_id','nunique')
)

items_agg["order_value"] = items_agg["products_value"] + items_agg["freight_value"]


# Paiements au niveau commande
payments_agg = silver_payments.groupby("order_id", as_index=False).agg(
    payment_total=("payment_value", "sum"),
    nb_payment_transactions=("payment_sequential", "count"),
    nb_payment_methods=("payment_type", "nunique"),
    max_installments=("payment_installments", "max"),
)


# Reviews au niveau commande
reviews_agg = silver_reviews.groupby("order_id", as_index=False).agg(
    review_score=("review_score", "mean")
)


In [139]:
gold_orders = (
    silver_orders.merge(items_agg, on="order_id", how="left", validate="one_to_one")
    .merge(payments_agg, on="order_id", how="left", validate="one_to_one")
    .merge(reviews_agg, on="order_id", how="left", validate="one_to_one")
    .merge(
        silver_customers[["customer_id", "customer_unique_id"]],
        on="customer_id",
        how="left",
        validate="many_to_one",
    )
)


In [140]:
gold_orders.columns


Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'dq_invalid_carrier_date', 'dq_invalid_delivery_sequence', 'nb_items',
       'products_value', 'freight_value', 'nb_sellers', 'order_value',
       'payment_total', 'nb_payment_transactions', 'nb_payment_methods',
       'max_installments', 'review_score', 'customer_unique_id'],
      dtype='str')

In [141]:
# Part du fret dans le montant total
gold_orders["freight_ratio"] = gold_orders["freight_value"] / gold_orders["order_value"]

# Prix moyen par article
gold_orders["avg_item_value"] = gold_orders["products_value"] / gold_orders["nb_items"]

# Commande multi-produits
gold_orders["is_multi_item"] = (
    gold_orders["nb_items"].gt(1).where(gold_orders["nb_items"].notna())
)

# Commande multi-vendeurs
gold_orders["is_multi_seller"] = (
    gold_orders["nb_sellers"].gt(1).where(gold_orders["nb_sellers"].notna())
)

# Paiement en plusieurs fois
gold_orders["is_installment_payment"] = (
    gold_orders["max_installments"].gt(1).where(gold_orders["max_installments"].notna())
)

# Plusieurs transactions de paiement
gold_orders["is_multi_payment"] = (
    gold_orders["nb_payment_transactions"]
    .gt(1)
    .where(gold_orders["nb_payment_transactions"].notna())
)

# Délai total de livraison
gold_orders["delivery_days"] = (
    gold_orders["order_delivered_customer_date"]
    - gold_orders["order_purchase_timestamp"]
).dt.days

# Temps entre achat et validation
gold_orders["approval_delay_hours"] = (
    gold_orders["order_approved_at"] - gold_orders["order_purchase_timestamp"]
).dt.total_seconds() / 3600

# Retard par rapport à la date estimée
gold_orders["delivery_delay_days"] = (
    gold_orders["order_delivered_customer_date"]
    - gold_orders["order_estimated_delivery_date"]
).dt.days

# Livraison en retard
gold_orders["is_late_delivery"] = (
    gold_orders["delivery_delay_days"]
    .gt(0)
    .where(gold_orders["delivery_delay_days"].notna())
)

# Livraison très en retard
gold_orders["is_very_late_delivery"] = (
    gold_orders["delivery_delay_days"]
    .gt(7)
    .where(gold_orders["delivery_delay_days"].notna())
)

# Livraison en avance
gold_orders["is_early_delivery"] = (
    gold_orders["delivery_delay_days"]
    .lt(0)
    .where(gold_orders["delivery_delay_days"].notna())
)

# Variables temporelles
gold_orders["purchase_year"] = gold_orders["order_purchase_timestamp"].dt.year

gold_orders["purchase_month"] = gold_orders["order_purchase_timestamp"].dt.month

gold_orders["purchase_dayofweek"] = gold_orders["order_purchase_timestamp"].dt.dayofweek

gold_orders["purchase_hour"] = gold_orders["order_purchase_timestamp"].dt.hour

# Week-end
gold_orders["is_weekend"] = gold_orders["purchase_dayofweek"] >= 5

# Moment de la journée
gold_orders["purchase_period"] = pd.cut(
    gold_orders["purchase_hour"],
    bins=[-1, 5, 11, 17, 23],
    labels=["night", "morning", "afternoon", "evening"],
)

# Avis positif
gold_orders["is_positive_review"] = (
    gold_orders["review_score"].ge(4).where(gold_orders["review_score"].notna())
)

# Avis négatif
gold_orders["is_negative_review"] = (
    gold_orders["review_score"].le(2).where(gold_orders["review_score"].notna())
)

# Commande à forte valeur
median_order_value = gold_orders["order_value"].median()

gold_orders["is_high_value_order"] = (
    gold_orders["order_value"]
    .gt(median_order_value)
    .where(gold_orders["order_value"].notna())
)


## Product

In [142]:
product_sales = silver_order_items.groupby("product_id", as_index=False).agg(
    nb_orders=("order_id", "nunique"),
    units_sold=("order_item_id", "count"),
    revenue=("price", "sum"),
    freight_revenue=("freight_value", "sum"),
    avg_selling_price=("price", "mean"),
    avg_freight=("freight_value", "mean"),
)


In [143]:
gold_products = silver_products.merge(
    product_sales, on="product_id", how="left", validate="one_to_one"
)


In [144]:
product_reviews = (
    silver_order_items[["order_id", "product_id"]]
    .merge(silver_reviews[["order_id", "review_score"]], on="order_id", how="left")
    .groupby("product_id", as_index=False)
    .agg(avg_review_score=("review_score", "mean"))
)


In [145]:
gold_products = gold_products.merge(
    product_reviews, on="product_id", how="left", validate="one_to_one"
)


In [146]:
gold_products.columns

Index(['product_id', 'product_name_length', 'product_category_name',
       'product_description_length', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm',
       'nb_orders', 'units_sold', 'revenue', 'freight_revenue',
       'avg_selling_price', 'avg_freight', 'avg_review_score'],
      dtype='str')

In [147]:
# ============================================================
# FEATURE ENGINEERING - GOLD_ORDERS
# ============================================================

# Valeur totale de la commande
gold_orders["order_value"] = (
    gold_orders["products_value"] + gold_orders["freight_value"]
)

# Part du fret dans la valeur totale
gold_orders["freight_ratio"] = gold_orders["freight_value"] / gold_orders["order_value"]

# Prix moyen par article
gold_orders["avg_item_value"] = gold_orders["products_value"] / gold_orders["nb_items"]

# Commande avec plusieurs articles
gold_orders["is_multi_item"] = (
    gold_orders["nb_items"].gt(1).where(gold_orders["nb_items"].notna())
)

# Commande avec plusieurs vendeurs
gold_orders["is_multi_seller"] = (
    gold_orders["nb_sellers"].gt(1).where(gold_orders["nb_sellers"].notna())
)

# Paiement en plusieurs fois
gold_orders["is_installment_payment"] = (
    gold_orders["max_installments"].gt(1).where(gold_orders["max_installments"].notna())
)

# Plusieurs transactions de paiement
gold_orders["is_multi_payment"] = (
    gold_orders["nb_payment_transactions"]
    .gt(1)
    .where(gold_orders["nb_payment_transactions"].notna())
)

# Délai total de livraison
gold_orders["delivery_days"] = (
    gold_orders["order_delivered_customer_date"]
    - gold_orders["order_purchase_timestamp"]
).dt.days

# Temps entre l'achat et l'approbation de la commande
gold_orders["approval_delay_hours"] = (
    gold_orders["order_approved_at"] - gold_orders["order_purchase_timestamp"]
).dt.total_seconds() / 3600

# Écart entre la livraison réelle et la livraison estimée
gold_orders["delivery_delay_days"] = (
    gold_orders["order_delivered_customer_date"]
    - gold_orders["order_estimated_delivery_date"]
).dt.days

# Livraison en retard
gold_orders["is_late_delivery"] = (
    gold_orders["delivery_delay_days"]
    .gt(0)
    .where(gold_orders["delivery_delay_days"].notna())
)

# Livraison très en retard (> 7 jours)
gold_orders["is_very_late_delivery"] = (
    gold_orders["delivery_delay_days"]
    .gt(7)
    .where(gold_orders["delivery_delay_days"].notna())
)

# Livraison en avance
gold_orders["is_early_delivery"] = (
    gold_orders["delivery_delay_days"]
    .lt(0)
    .where(gold_orders["delivery_delay_days"].notna())
)

# Variables temporelles d'achat
gold_orders["purchase_year"] = gold_orders["order_purchase_timestamp"].dt.year

gold_orders["purchase_month"] = gold_orders["order_purchase_timestamp"].dt.month

gold_orders["purchase_quarter"] = gold_orders["order_purchase_timestamp"].dt.quarter

gold_orders["purchase_dayofweek"] = gold_orders["order_purchase_timestamp"].dt.dayofweek

gold_orders["purchase_day_name"] = gold_orders["order_purchase_timestamp"].dt.day_name()

gold_orders["purchase_hour"] = gold_orders["order_purchase_timestamp"].dt.hour

# Achat effectué le week-end
gold_orders["is_weekend"] = gold_orders["purchase_dayofweek"] >= 5

# Moment de la journée
gold_orders["purchase_period"] = pd.cut(
    gold_orders["purchase_hour"],
    bins=[-1, 5, 11, 17, 23],
    labels=["night", "morning", "afternoon", "evening"],
)

# Avis positif
gold_orders["is_positive_review"] = (
    gold_orders["review_score"].ge(4).where(gold_orders["review_score"].notna())
)

# Avis négatif
gold_orders["is_negative_review"] = (
    gold_orders["review_score"].le(2).where(gold_orders["review_score"].notna())
)

# Seuil pour définir une commande à forte valeur
median_order_value = gold_orders["order_value"].median()

# Commande à forte valeur
gold_orders["is_high_value_order"] = (
    gold_orders["order_value"]
    .gt(median_order_value)
    .where(gold_orders["order_value"].notna())
)


### Gold Customer Table

In [148]:
# ============================================================
# GOLD CUSTOMERS
# Grain : 1 ligne = 1 customer_unique_id
# ============================================================

# Agrégation principale
gold_customers = (
    gold_orders
    .groupby("customer_unique_id", as_index=False)
    .agg(
        nb_orders=("order_id", "nunique"),
        # total_spent=("order_value", "sum"),
        avg_order_value=("order_value", "mean"),
        min_order_value=("order_value", "min"),
        max_order_value=("order_value", "max"),
        total_spent=("order_value", lambda x: x.sum(min_count=1)),
        total_freight=("freight_value", lambda x: x.sum(min_count=1)),
        
        # total_freight=("freight_value", "sum"),
        avg_freight=("freight_value", "mean"),

        total_items=("nb_items", "sum"),
        avg_items_per_order=("nb_items", "mean"),

        first_order_date=("order_purchase_timestamp", "min"),
        last_order_date=("order_purchase_timestamp", "max"),

        avg_review_score=("review_score", "mean"),
        avg_delivery_days=("delivery_days", "mean"),
        avg_delivery_delay_days=("delivery_delay_days", "mean"),

        nb_late_orders=("is_late_delivery", "sum"),
        nb_positive_reviews=("is_positive_review", "sum"),
        nb_negative_reviews=("is_negative_review", "sum"),

        avg_installments=("max_installments", "mean"),
        nb_multi_payment_orders=("is_multi_payment", "sum")
    )
)

In [149]:
# Date de référence
reference_date = gold_orders["order_purchase_timestamp"].max()

# Récence : nombre de jours depuis le dernier achat
gold_customers["recency_days"] = (
    reference_date - gold_customers["last_order_date"]
).dt.days

# Ancienneté client
gold_customers["customer_age_days"] = (
    gold_customers["last_order_date"] - gold_customers["first_order_date"]
).dt.days

# Fréquence d'achat
gold_customers["purchase_frequency"] = gold_customers["nb_orders"] / gold_customers[
    "customer_age_days"
].replace(0, 1)

# Nombre moyen d'articles par commande
gold_customers["avg_items_per_order"] = (
    gold_customers["total_items"] / gold_customers["nb_orders"]
)

# Dépense moyenne par article
gold_customers["avg_spend_per_item"] = (
    gold_customers["total_spent"] / gold_customers["total_items"]
)

# Part moyenne du fret
gold_customers["freight_ratio"] = (
    gold_customers["total_freight"] / gold_customers["total_spent"]
)

# Client récurrent
gold_customers["is_repeat_customer"] = gold_customers["nb_orders"] > 1

# Taux de commandes en retard
gold_customers["late_order_rate"] = (
    gold_customers["nb_late_orders"] / gold_customers["nb_orders"]
)

# Taux d'avis positifs
gold_customers["positive_review_rate"] = (
    gold_customers["nb_positive_reviews"] / gold_customers["nb_orders"]
)

# Taux d'avis négatifs
gold_customers["negative_review_rate"] = (
    gold_customers["nb_negative_reviews"] / gold_customers["nb_orders"]
)

# Client à forte valeur
median_customer_value = gold_customers["total_spent"].median()

gold_customers["is_high_value_customer"] = (
    gold_customers["total_spent"]
    .gt(median_customer_value)
    .where(gold_customers["total_spent"].notna())
)

# Client fidèle : exemple simple
gold_customers["is_loyal_customer"] = gold_customers["nb_orders"] >= 2

# Client potentiellement insatisfait
gold_customers["is_unsatisfied_customer"] = (
    gold_customers["avg_review_score"]
    .lt(3)
    .where(gold_customers["avg_review_score"].notna())
)

# Client avec problèmes logistiques fréquents
gold_customers["has_logistics_issues"] = gold_customers["late_order_rate"] > 0.5


In [150]:
silver_customers

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_state,customer_city
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,SP,franca
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,SP,sao bernardo do campo
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,SP,sao paulo
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,SP,mogi das cruzes
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,SP,campinas
...,...,...,...,...,...
99436,17ddf5dd5d51696bb3d7c6291687be6f,1a29b476fee25c95fbafc67c5ac95cf8,03937,SP,sao paulo
99437,e7b71a9017aa05c9a7fd292d714858e8,d52a67c98be1cf6a5c84435bd38d095d,06764,SP,taboao da serra
99438,5e28dfe12db7fb50a4b2f691faecea5e,e9f50caf99f032f0bf3c55141f019d99,60115,CE,fortaleza
99439,56b18e2166679b8a959d72dd06da27f9,73c2643a0a458b49f58cea58833b192e,92120,RS,canoas


In [151]:
customer_geo = silver_customers[
    [
        "customer_unique_id",
        "customer_city",
        "customer_state",
        "customer_zip_code_prefix",
    ]
].drop_duplicates("customer_unique_id")

gold_customers = gold_customers.merge(
    customer_geo, on="customer_unique_id", how="left", validate="one_to_one"
)


In [152]:
print(gold_products.shape)
print(gold_customers.shape)
print(gold_orders.shape)


(32951, 16)
(96096, 36)
(99441, 44)


In [153]:
WIDTH = 100


def print_table_info(title, df):
    print("=" * WIDTH)
    print(title.center(WIDTH))
    print("=" * WIDTH)
    print()
    df.info()
    print()


print_table_info("PRODUCTS TABLE", gold_products)
print_table_info("ORDERS TABLE", gold_orders)
print_table_info("CUSTOMERS TABLE", gold_customers)


                                           PRODUCTS TABLE                                           

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_name_length         32341 non-null  float64
 2   product_category_name       32951 non-null  str    
 3   product_description_length  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32945 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
 9   nb_orders                   32951 non-null  int64  
 10  units_sold                  32951 non-null  int64  
 11  revenue                     32951 non-null  float64
 1

In [154]:
bool_cols_orders = gold_orders.columns[gold_orders.columns.str.startswith("is_")]
gold_orders[bool_cols_orders] = gold_orders[bool_cols_orders].astype("boolean")


In [155]:
for col in [
    "nb_items",
    "nb_sellers",
    "nb_payment_transactions",
    "nb_payment_methods",
    "max_installments",
]:
    gold_orders[col] = gold_orders[col].astype("Int64")


In [156]:
int_cols_customers = [
    "nb_late_orders",
    "nb_positive_reviews",
    "nb_negative_reviews",
]
rate_cols = [
    "late_order_rate",
    "positive_review_rate",
    "negative_review_rate",
]

gold_customers[rate_cols] = gold_customers[rate_cols].astype("Float64")
gold_customers[int_cols_customers] = gold_customers[int_cols_customers].astype("Int64")
gold_customers["is_unsatisfied_customer"] = gold_customers[
    "is_unsatisfied_customer"
].astype("boolean")


gold_customers["nb_multi_payment_orders"] = gold_customers[
    "nb_multi_payment_orders"
].astype("Int64")

gold_customers["is_high_value_customer"] = gold_customers[
    "is_high_value_customer"
].astype("boolean")


gold_customers["total_items"] = gold_customers["total_items"].astype("Int64")


In [157]:
for name, df in {
    "products": gold_products,
    "orders": gold_orders,
    "customers": gold_customers,
}.items():
    print(f"\n{name.upper()}")
    print("Rows:", len(df))
    print("Duplicates:", df.duplicated().sum())
    print("Missing %:")
    print((df.isna().mean() * 100).sort_values(ascending=False).head(10))



PRODUCTS
Rows: 32951
Duplicates: 0
Missing %:
product_name_length           1.851234
product_description_length    1.851234
product_photos_qty            1.851234
avg_review_score              0.491639
product_weight_g              0.018209
product_length_cm             0.006070
product_width_cm              0.006070
product_height_cm             0.006070
product_id                    0.000000
product_category_name         0.000000
dtype: float64

ORDERS
Rows: 99441
Duplicates: 0
Missing %:
order_delivered_customer_date    2.981668
is_late_delivery                 2.981668
is_very_late_delivery            2.981668
delivery_days                    2.981668
delivery_delay_days              2.981668
is_early_delivery                2.981668
order_delivered_carrier_date     1.793023
freight_value                    0.779357
nb_sellers                       0.779357
order_value                      0.779357
dtype: float64

CUSTOMERS
Rows: 96096
Duplicates: 0
Missing %:
avg_delivery_days   

In [158]:
# Produits sans review
assert (
    gold_products.loc[gold_products["avg_review_score"].isna(), "product_id"]
    .notna()
    .all()
)

# Commandes sans delivery_days => pas de date de livraison client
assert (
    gold_orders["delivery_days"].isna()
    == gold_orders["order_delivered_customer_date"].isna()
).all()

# Commandes sans order_value => pas d'agrégat order_items
assert (gold_orders["order_value"].isna() == gold_orders["nb_items"].isna()).all()

assert gold_orders["order_id"].is_unique
assert gold_products["product_id"].is_unique
assert gold_customers["customer_unique_id"].is_unique

assert (gold_orders["order_value"].dropna() >= 0).all()
assert (gold_products["revenue"] >= 0).all()
assert (gold_customers["total_spent"].dropna() >= 0).all()


In [159]:
print_table_info("PRODUCTS TABLE", gold_products)
print_table_info("ORDERS TABLE", gold_orders)
print_table_info("CUSTOMERS TABLE", gold_customers)


                                           PRODUCTS TABLE                                           

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_name_length         32341 non-null  float64
 2   product_category_name       32951 non-null  str    
 3   product_description_length  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32945 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
 9   nb_orders                   32951 non-null  int64  
 10  units_sold                  32951 non-null  int64  
 11  revenue                     32951 non-null  float64
 1

In [160]:
gold_orders.to_parquet(r"..\data\gold\olist_orders.parquet")
gold_products.to_parquet(r"..\data\gold\olist_products.parquet")
gold_customers.to_parquet(r"..\data\gold\olist_customers.parquet")